In [ ]:
import atexit
import hashlib
import json
import os
import threading
import time
from pathlib import Path

RUN_ID = "hoct-multibackbone-probe-v1"
STARTED = time.monotonic()
FINISHED = False
TERMINAL = Path("/kaggle/working/launcher_terminal.json")


def write_terminal(status, error=None):
    training = Path("/kaggle/working/hoct_multibackbone_v1/training_terminal.json")
    payload = {
        "run_id": RUN_ID,
        "status": status,
        "elapsed_seconds": round(time.monotonic() - STARTED, 3),
        "declared_budget_seconds": 14400,
        "hard_stop_seconds": 14100,
        "training_terminal_exists": training.is_file(),
        "submission_created": Path("/kaggle/working/submission.csv").is_file(),
    }
    if error:
        payload["error"] = str(error)
    if training.is_file():
        payload["training_terminal_sha256"] = hashlib.sha256(training.read_bytes()).hexdigest()
    temporary = TERMINAL.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(TERMINAL)


def abort_terminal():
    if not FINISHED:
        write_terminal("aborted")


def budget_expired():
    write_terminal("budget_expired")
    os._exit(124)


atexit.register(abort_terminal)
TIMER = threading.Timer(14100, budget_expired)
TIMER.daemon = True
TIMER.start()
print("HOCT multibackbone watchdog armed for 14,100 seconds.")


# Biohub HOCT multi-backbone association adaptation

This private experiment fits independent linear probes on the official general_v1 and CTC-specialized ctc_v0 HOCT backbones. It selects among four single-model heads and eight ensemble variants using two complete movies, then reads two disjoint acceptance movies once. It creates no competition submission and uses no leaderboard feedback. When available, it reuses the preceding run's hash-verified comparator topology to reserve the GPU window for HOCT feature extraction.


In [ ]:
import importlib
import shutil
import subprocess
import sys
import zipfile


def first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)


input_root = Path("/kaggle/input")
hoct_runtime = first_existing([
    Path("/kaggle/input/datasets/indarkarhana/biohub-hoct-multibackbone-runtime-v1"),
    Path("/kaggle/input/biohub-hoct-multibackbone-runtime-v1"),
])
graph_runtime = first_existing([
    Path("/kaggle/input/datasets/indarkarhana/biohub-trackastra-graph-runtime-v1"),
    Path("/kaggle/input/biohub-trackastra-graph-runtime-v1"),
])
competition = first_existing([
    Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
    Path("/kaggle/input/biohub-cell-tracking-during-development"),
])
support = first_existing([
    Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"),
    Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1"),
])
deepcenter = first_existing([
    Path("/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1"),
    Path("/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1"),
])
trackastra_validation_output = first_existing([
    Path("/kaggle/input/biohub-trackastra-raw-confidence-acceptance-v2"),
    Path("/kaggle/input/kernels/indarkarhana/biohub-trackastra-raw-confidence-acceptance-v2"),
])
if None in (hoct_runtime, graph_runtime, competition, support, deepcenter):
    raise FileNotFoundError({
        "hoct_runtime": hoct_runtime,
        "graph_runtime": graph_runtime,
        "competition": competition,
        "support": support,
        "deepcenter": deepcenter,
    })


def materialize_root(runtime, name):
    bundle = runtime / "runtime_bundle.zip"
    if not bundle.is_file():
        return runtime
    destination = Path("/kaggle/working") / f"{name}_runtime_bundle"
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with zipfile.ZipFile(bundle) as handle:
        handle.extractall(destination)
    return destination


hoct_runtime = materialize_root(hoct_runtime, "hoct")
graph_runtime = materialize_root(graph_runtime, "graph")


def materialize_runtime_directory(runtime, name, marker):
    direct = runtime / name
    if (direct / marker).exists():
        return direct
    archive = runtime / f"{name}.zip"
    if not archive.is_file():
        raise FileNotFoundError(f"Missing runtime directory/archive: {name}")
    destination = Path("/kaggle/working/runtime_materialized") / name
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
    for candidate in (destination, destination / name):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Archive {archive} did not contain {marker}")


validation_dir = materialize_runtime_directory(
    graph_runtime, "validator_raw", "44b6_12dfb391.geff/zarr.json"
)
wheel_dirs = sorted({path.parent for path in support.rglob("*.whl")})
if not wheel_dirs:
    raise FileNotFoundError(f"No offline wheels found below {support}")
numpy_before = importlib.import_module("numpy").__version__
if numpy_before != "2.0.2":
    raise RuntimeError(f"Unexpected Kaggle NumPy: {numpy_before}")
pip_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
for wheel_dir in wheel_dirs:
    pip_cmd.extend(["--find-links", str(wheel_dir)])
pip_cmd.extend([
    "bidict==0.23.1", "donfig==0.8.1.post1", "geff==1.2.0.1.1",
    "geff-spec==1.1.1", "ilpy==0.6.0", "numcodecs==0.15.1",
    "polars==1.42.0", "polars-runtime-32==1.42.0", "pyscipopt==6.2.1",
    "rustworkx==0.18.0", "tracksdata==0.1.0rc6.dev3+g980c2d30a",
    "zarr==3.2.1",
])
subprocess.run(pip_cmd, check=True)
for module in ("numpy", "scipy", "torch", "tracksdata", "zarr", "polars", "dask", "yaml"):
    importlib.import_module(module)
if importlib.import_module("numpy").__version__ != numpy_before:
    raise RuntimeError("Offline dependency install changed NumPy")
if not importlib.import_module("torch").cuda.is_available():
    raise RuntimeError("CUDA is unavailable")

hoct_manifest = json.loads((hoct_runtime / "SOURCE_MANIFEST.json").read_text(encoding="utf-8"))
for name in (
    "association_ensemble.py", "biohub_adapter.py", "multibackbone.py",
    "train_biohub_hoct_probe.py", "train_biohub_hoct_multibackbone.py",
    "general_v1.pt", "ctc_v0.pt",
):
    actual = hashlib.sha256((hoct_runtime / name).read_bytes()).hexdigest()
    if actual != hoct_manifest["files"][name]["sha256"]:
        raise RuntimeError(f"HOCT runtime hash mismatch: {name}")
graph_manifest = json.loads((graph_runtime / "SOURCE_MANIFEST.json").read_text(encoding="utf-8"))
for name in (
    "trainer.py", "hybrid_linker.py", "rerank_submission.py",
    "materialize_public_validation.py", "public_preset_source.py", "public_config_source.py",
    "public_postprocess_source.py",
):
    actual = hashlib.sha256((graph_runtime / name).read_bytes()).hexdigest()
    if actual != graph_manifest["files"][name]["sha256"]:
        raise RuntimeError(f"Graph runtime hash mismatch: {name}")
print(json.dumps({
    "hoct_runtime": str(hoct_runtime),
    "graph_runtime": str(graph_runtime),
    "competition": str(competition),
    "validation_dir": str(validation_dir),
    "deepcenter": str(deepcenter),
    "source_model_sha256": {name: value["sha256"] for name, value in hoct_manifest["pretrained_models"].items()},
}, indent=2))


In [ ]:
EXPECTED_PROCESSED_NODES = {
    "44b6_12dfb391": 44139,
    "44b6_267148e4": 21768,
    "6bba_062c8d37": 5812,
    "6bba_07e24132": 26204,
}
EXPECTED_DEEPCENTER_SHA256 = "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0"


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def validated_cached_topology(root):
    if root is None:
        return None
    launcher_candidates = list(root.rglob("launcher_terminal.json"))
    if not launcher_candidates:
        return None
    valid_launcher = False
    for path in launcher_candidates:
        try:
            terminal = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if (
            terminal.get("run_id") == "trackastra-raw-confidence-acceptance-v2"
            and terminal.get("status") in {"completed", "failed", "aborted", "budget_expired"}
            and terminal.get("submission_created") is False
        ):
            valid_launcher = True
            break
    if not valid_launcher:
        return None
    for report_path in sorted(root.rglob("processed_validation_report.json")):
        csv_path = report_path.with_name("processed_validation.csv")
        if not csv_path.is_file():
            continue
        try:
            report = json.loads(report_path.read_text(encoding="utf-8"))
            csv_sha256 = sha256_file(csv_path)
        except (OSError, ValueError):
            continue
        expected_hashes = {
            "public_preset_source_sha256": graph_manifest["files"]["public_preset_source.py"]["sha256"],
            "public_config_source_sha256": graph_manifest["files"]["public_config_source.py"]["sha256"],
            "public_postprocess_source_sha256": graph_manifest["files"]["public_postprocess_source.py"]["sha256"],
        }
        checks = [
            report.get("schema_version") == 1,
            report.get("status") == "completed",
            report.get("source_notebook") == "evgendvorkin/biohub-0-927-lb",
            report.get("role") == "frozen comparator postprocessing only",
            report.get("ground_truth_read_for_postprocessing") is False,
            report.get("public_leaderboard_used_for_selection") is False,
            report.get("processed_validation_sha256") == csv_sha256,
            report.get("deepcenter_checkpoint", {}).get("sha256") == EXPECTED_DEEPCENTER_SHA256,
            report.get("deepcenter_checkpoint", {}).get("expected_epoch") == 2,
            all(report.get(key) == value for key, value in expected_hashes.items()),
            all(
                report.get("datasets", {}).get(stem, {}).get("processed_nodes") == count
                and report.get("datasets", {}).get(stem, {}).get("expected_processed_nodes") == count
                for stem, count in EXPECTED_PROCESSED_NODES.items()
            ),
        ]
        if all(checks):
            print(json.dumps({
                "processed_topology_source": "hash_verified_trackastra_v2_kernel_output",
                "processed_validation_csv": str(csv_path),
                "processed_validation_sha256": report["processed_validation_sha256"],
            }, indent=2))
            return csv_path
    return None


cached_processed_csv = validated_cached_topology(trackastra_validation_output)
if cached_processed_csv is None:
    print("No valid cached topology found; using the exact in-kernel materializer.")

processed_dir = Path("/kaggle/working/processed_validation")
materialize_command = [
    sys.executable,
    str(graph_runtime / "materialize_public_validation.py"),
    "--raw-validation-root", str(validation_dir),
    "--competition-dir", str(competition),
    "--public-preset-source", str(graph_runtime / "public_preset_source.py"),
    "--public-config-source", str(graph_runtime / "public_config_source.py"),
    "--public-postprocess-source", str(graph_runtime / "public_postprocess_source.py"),
    "--output-dir", str(processed_dir),
]
if cached_processed_csv is None:
    print("Materializing final-topology clean validation:", " ".join(materialize_command))
    try:
        subprocess.run(materialize_command, check=True)
    except Exception as exc:
        write_terminal("failed", exc)
        raise
    processed_validation_csv = processed_dir / "processed_validation.csv"
else:
    processed_validation_csv = cached_processed_csv

output_dir = Path("/kaggle/working/hoct_multibackbone_v1")
command = [
    sys.executable,
    str(hoct_runtime / "train_biohub_hoct_multibackbone.py"),
    "--competition-dir", str(competition),
    "--general-model", str(hoct_runtime / "general_v1.pt"),
    "--ctc-model", str(hoct_runtime / "ctc_v0.pt"),
    "--validation-predictions", str(validation_dir),
    "--processed-validation-csv", str(processed_validation_csv),
    "--output-dir", str(output_dir),
    "--train-per-prefix", "128",
    "--core-size", "128",
    "--max-wall-seconds", "13200",
]
environment = os.environ.copy()
environment["PYTHONPATH"] = os.pathsep.join([str(hoct_runtime), str(graph_runtime)])
print("Launching Biohub HOCT multibackbone probe:", " ".join(command))
try:
    subprocess.run(command, check=True, env=environment)
except Exception as exc:
    write_terminal("failed", exc)
    raise

training = json.loads((output_dir / "training_terminal.json").read_text(encoding="utf-8"))
if Path("/kaggle/working/submission.csv").exists():
    raise RuntimeError("Training unexpectedly created a submission")
print(json.dumps({
    "training_evidence": training["training_evidence"],
    "selected": training["selected"],
    "selection_delta_vs_base": training["selection_delta_vs_base"],
    "acceptance_delta_vs_base": training["acceptance_delta_vs_base"],
    "association_acceptance_passed": training["association_acceptance_passed"],
}, indent=2))


In [ ]:
FINISHED = True
TIMER.cancel()
write_terminal("completed")
print("HOCT multibackbone experiment complete; no submission was created.")
